Imports

In [1]:
import pandas as pd
import numpy as np

Load data

In [57]:
DEMO_J = pd.read_sas("./raw/2017_2018/DEMO_J.XPT")
BPX_J = pd.read_sas("./raw/2017_2018/BPX_J.XPT")
BMX_J = pd.read_sas("./raw/2017_2018/BMX_J.XPT")
CBC_J = pd.read_sas("./raw/2017_2018/CBC_J.XPT")
FERTIN_J = pd.read_sas("./raw/2017_2018/FERTIN_J.XPT")
VID_J = pd.read_sas("./raw/2017_2018/VID_J.XPT")
HSCRP_J = pd.read_sas("./raw/2017_2018/HSCRP_J.XPT")
BIOPRO_J = pd.read_sas("./raw/2017_2018/BIOPRO_J.XPT")
PAQ_J = pd.read_sas("./raw/2017_2018/PAQ_J.XPT")
ALQ_J = pd.read_sas("./raw/2017_2018/ALQ_J.XPT")
SMQ_J = pd.read_sas("./raw/2017_2018/SMQ_J.XPT")
SLQ_J = pd.read_sas("./raw/2017_2018/SLQ_J.XPT")
DUQ_J = pd.read_sas("./raw/2017_2018/DUQ_J.XPT")
WHQ_J = pd.read_sas("./raw/2017_2018/WHQ_J.XPT")
SMQRTU_J = pd.read_sas("./raw/2017_2018/SMQRTU_J.XPT")
DPQ_J = pd.read_sas("./raw/2017_2018/DPQ_J.XPT")

data_frames = {
    'DEMO': DEMO_J,
    'BPX': BPX_J,
    'BMX': BMX_J,
    'CBC': CBC_J,
    'FERTIN': FERTIN_J,
    'VID': VID_J,
    'HSCRP': HSCRP_J,
    'BIOPRO': BIOPRO_J,
    'PAQ': PAQ_J,
    'ALQ': ALQ_J,
    'SMQ': SMQ_J,
    'SMQRTU': SMQRTU_J,
    'DPQ': DPQ_J,
    'SLQ': SLQ_J,
    'DUQ': DUQ_J,
    'WHQ': WHQ_J
}



In [81]:
# codebook includes a description of each data file and variable, as well as a first pass at selecting model input and output variables

codebook = pd.read_csv('./raw/2017_2018/codebook_2017_2018.csv')
codebook.head()

,Variable Name,Variable Description,Data File Name,Data File Description,Begin Year,EndYear,Component,Use Constraints,Model Feature,Model Output
0,AIALANGA,Language of the MEC ACASI Interview Instrument,DEMO_J,Demographic Variables and Sample Weights,2017,2018,Demographics,NaN,NaN,NaN
1,DMDBORN4,In what country {were you/was SP} born?,DEMO_J,Demographic Variables and Sample Weights,2017,2018,Demographics,NaN,NaN,NaN
2,DMDCITZN,{Are you/Is SP} a citizen of the United States...,DEMO_J,Demographic Variables and Sample Weights,2017,2018,Demographics,NaN,NaN,NaN
3,DMDEDUC2,What is the highest grade or level of school {...,DEMO_J,Demographic Variables and Sample Weights,2017,2018,Demographics,NaN,NaN,NaN
4,DMDEDUC3,What is the highest grade or level of school {...,DEMO_J,Demographic Variables and Sample Weights,2017,2018,Demographics,NaN,NaN,NaN


In [82]:
for name, val in data_frames.items():
    print(codebook[codebook['Data File Name'] == f"{name}_J"]['Data File Description'].iloc[0])
    print('Shape: ', val.shape)
    print('n patients: ', val['SEQN'].nunique())
    print()

Demographic Variables and Sample Weights
Shape:  (9254, 46)
n patients:  9254

Blood Pressure 
Shape:  (8704, 21)
n patients:  8704

Body Measures
Shape:  (8704, 21)
n patients:  8704

Complete Blood Count with 5-Part Differential
Shape:  (8366, 22)
n patients:  8366

Ferritin 
Shape:  (7332, 3)
n patients:  7332

Vitamin D 
Shape:  (8366, 9)
n patients:  8366

High-Sensitivity C-Reactive Protein
Shape:  (8366, 3)
n patients:  8366

Standard Biochemistry Profile
Shape:  (6401, 41)
n patients:  6401

Physical Activity
Shape:  (5856, 17)
n patients:  5856

Alcohol Use
Shape:  (5533, 10)
n patients:  5533

Smoking - Cigarette Use
Shape:  (6724, 37)
n patients:  6724

Smoking - Recent Tobacco Use
Shape:  (6401, 27)
n patients:  6401

Mental Health - Depression Screener
Shape:  (5533, 11)
n patients:  5533

Sleep Disorders
Shape:  (6161, 11)
n patients:  6161

Drug Use
Shape:  (4572, 41)
n patients:  4572

Weight History
Shape:  (6161, 37)
n patients:  6161



In [83]:
from functools import reduce

# Prefix non-SEQN columns to avoid name collisions, then outer-merge all on SEQN
dfs = [df.rename(columns=lambda c: c if c == 'SEQN' else f"{name}_{c}") for name, df in data_frames.items()]
merged = reduce(lambda left, right: pd.merge(left, right, on='SEQN', how='outer'), dfs)
merged = merged.replace(5.397605346934028e-79, 0) # weird things happening when value = 0 in sas files

print(merged.shape)
print(merged['SEQN'].nunique())

(9254, 342)
9254


In [84]:
# select columns from `merged` where codebook marks them as model feature or model output
def _find_col(codebook, keywords):
    for c in codebook.columns:
        low = c.lower()
        if all(k in low for k in keywords):
            return c
    for c in codebook.columns:
        low = c.lower()
        if any(k in low for k in keywords):
            return c
    return None

var_col = _find_col(codebook, ['var']) or _find_col(codebook, ['name']) or _find_col(codebook, ['column'])
feat_col = _find_col(codebook, ['model','feature']) or _find_col(codebook, ['feature'])
out_col = _find_col(codebook, ['model','output']) or _find_col(codebook, ['output'])
dfname_col = _find_col(codebook, ['data file']) or _find_col(codebook, ['file'])

if var_col is None:
    raise ValueError("Couldn't locate variable name column in codebook.")

# build boolean mask for rows flagged as feature or output (accepts True/False, 1/0, 'true'/'1', etc.)
masks = []
for col in (feat_col, out_col):
    if col is not None:
        s = codebook[col].astype(str).str.lower()
        masks.append(s.isin(['1','true','t','y','yes'] ) | (pd.to_numeric(codebook[col], errors='coerce')>0).fillna(False))
if not masks:
    raise ValueError("Couldn't find feature/output flag columns in codebook.")
mask = masks[0]
for m in masks[1:]:
    mask = mask | m

selected_rows = codebook.loc[mask].dropna(subset=[var_col])

# construct merged column names: prefix from data file name (e.g. 'WHQ_J' -> 'WHQ') + '_' + variable
def merged_col_name(row):
    var = str(row[var_col]).strip()
    if var.upper() == 'SEQN':
        return 'SEQN'
    dfname = ''
    if dfname_col and pd.notna(row.get(dfname_col)):
        dfname = str(row[dfname_col])
    prefix = dfname.replace('_J','').rstrip('_') if dfname else ''
    return f"{prefix}_{var}" if prefix else var

cols = [merged_col_name(r) for _, r in selected_rows.iterrows()]
cols = ['SEQN'] + [c for c in cols if c in merged.columns and c != 'SEQN']

merged_selected = merged[cols].copy()
merged_selected.shape

(9254, 99)

In [85]:
merged_selected.head()

,SEQN,DEMO_RIAGENDR,DEMO_RIDAGEYR,DEMO_RIDEXPRG,DEMO_RIDRETH3,DEMO_WTINT2YR,DEMO_WTMEC2YR,BPX_BPXCHR,BPX_BPXDI1,BPX_BPXDI2,...,DUQ_DUQ250,DUQ_DUQ280,DUQ_DUQ290,DUQ_DUQ320,DUQ_DUQ330,DUQ_DUQ360,ALQ_ALQ170,ALQ_ALQ270,ALQ_ALQ280,ALQ_ALQ290
0,93703.0,2.0,2.0,NaN,6.0,9246.491865,8539.731348,120.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,1.0,2.0,NaN,3.0,37338.768343,42566.614750,114.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,2.0,66.0,NaN,4.0,8614.571172,8338.419786,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN
3,93706.0,1.0,18.0,NaN,6.0,8548.632619,8723.439814,NaN,74.0,70.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,93707.0,1.0,13.0,NaN,7.0,6769.344567,7064.609730,NaN,38.0,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# how many participants completed all questions of the depression screener (model output)

output_cols = list(codebook[codebook['Model Output'] == True]['Variable Name'])
output_cols = [f"DPQ_{col}" if not col.startswith("DPQ_") else col for col in output_cols]

merged_outputs = merged_selected[['SEQN'] + [col for col in output_cols if col in merged_selected.columns]]

# drop refusals and missing surveys
merged_outputs = merged_outputs[(merged_outputs[output_cols] <= 3).all(axis=1)]
merged_outputs.dropna(subset=output_cols, how='all').shape

In [88]:
merged_outputs.dropna(subset=output_cols, how='all').shape

(5094, 10)